In [ ]:
# Install required libraries if needed
!pip install pandas-datareader matplotlib seaborn

In [2]:
import pandas as pd
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [3]:
# 1. Adjust timeframe to start at the post-WWII baseline (1945)
start_date = datetime(1945, 1, 1)
end_date = datetime(2026, 1, 1)

In [ ]:
# Metrics mapping dictionary
series_dict = {
    'Birth_Rate_Per_1000': 'SPDYNCBRTINUSA',
    'Real_GDP_Per_Capita': 'A939RX0Q048SBEA',
    'Household_Net_Worth': 'HNOTNWA027S'
}
start_year = 1945

df_list = []

print("Fetching and verifying data from FRED...")
for name, series_id in series_dict.items():
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"

    # Read the data, feeding a browser User-Agent
    data = pd.read_csv(
        url,
        na_values='.',
        storage_options={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    )

    # Clean whitespace strings from the columns list
    data.columns = data.columns.str.strip()

    date_col = data.columns[0]
    value_col = data.columns[1]

    # Convert dates safely
    data[date_col] = pd.to_datetime(data[date_col])

    # CRITICAL ADDITION: Coerce value columns to float types, turning any weird symbols into NaN
    data[value_col] = pd.to_numeric(data[value_col], errors='coerce')

    # Isolate data and establish index
    data.set_index(date_col, inplace=True)
    data = data[[value_col]]
    data.columns = [name]

    df_list.append(data)

# Combine historical vectors
df_raw = pd.concat(df_list, axis=1)

# Apply baseline year filter
df_raw = df_raw[df_raw.index.year >= start_year]

# Clean up timeframe spacing with the modern annual resample 'YE' (Year End)
df_annual = df_raw.resample('YE').mean()
df_annual.index = df_annual.index.year
df_clean = df_annual.dropna()

print("\nCode Check Complete! Clean dataframe constructed successfully:")
print(df_clean.head())

Fetching and verifying data from FRED...


In [ ]:
# 3. Clean and Resample Data
# Using 'YE' (Year End) to align with modern pandas standards
df_annual = df_raw.resample('YE').mean()
df_annual.index = df_annual.index.year  # Convert index to simple Year integers
df_clean = df_annual.dropna()           # Drop rows missing any of the 3 metrics

print("\nData Sample (First 5 Years):")
print(df_clean.head())

In [ ]:
# 4. Visualization: Two Plots Side-by-Side for Comparison
sns.set_theme(style="whitegrid")
fig, (ax1, ax3) = plt.subplots(1, 2, figsize=(16, 6))

# --- PLOT 1: Birth Rate vs. Income (Real GDP) ---
color_birth = '#1f77b4'
ax1.set_xlabel('Year', fontweight='bold')
ax1.set_ylabel('Crude Birth Rate (per 1,000)', color=color_birth, fontweight='bold')
line1 = ax1.plot(df_clean.index, df_clean['Birth_Rate_Per_1000'], color=color_birth, linewidth=2.5, label='Birth Rate')
ax1.tick_params(axis='y', labelcolor=color_birth)

ax2 = ax1.twinx()
color_income = '#2ca02c'
ax2.set_ylabel('Real GDP per Capita (USD)', color=color_income, fontweight='bold')
line2 = ax2.plot(df_clean.index, df_clean['Real_GDP_Per_Capita'], color=color_income, linewidth=2.5, linestyle='--', label='Real GDP Per Capita')
ax2.tick_params(axis='y', labelcolor=color_income)

ax1.set_title('Birth Rate vs. Real Income (Post-1945)', fontsize=12, fontweight='bold')

# --- PLOT 2: Birth Rate vs. Aggregate Household Net Worth ---
ax3.set_xlabel('Year', fontweight='bold')
ax3.set_ylabel('Crude Birth Rate (per 1,000)', color=color_birth, fontweight='bold')
line3 = ax3.plot(df_clean.index, df_clean['Birth_Rate_Per_1000'], color=color_birth, linewidth=2.5, label='Birth Rate')
ax3.tick_params(axis='y', labelcolor=color_birth)

ax4 = ax3.twinx()
color_wealth = '#9467bd'  # Purple for wealth
ax4.set_ylabel('Household Net Worth (Billions of USD)', color=color_wealth, fontweight='bold')
line4 = ax4.plot(df_clean.index, df_clean['Household_Net_Worth'], color=color_wealth, linewidth=2.5, linestyle=':', label='Household Net Worth')
ax4.tick_params(axis='y', labelcolor=color_wealth)

ax3.set_title('Birth Rate vs. Household Net Worth (Post-1945)', fontsize=12, fontweight='bold')

plt.suptitle('U.S. Demographic Shifts vs. National Wealth Metrics', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 5. Output Correlation Summary
print("\n--- Correlation Matrix ---")
print(df_clean.corr()['Birth_Rate_Per_1000'])